In [1]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [2]:
date = read_table("select * from sc_gold.dim_date")
state= read_table("select * from sc_gold.dim_state")

In [3]:
df = read_table("select * from sc_silver.forecast_gdp_unemp")
df 

,state,date,gdp_growth_yoy,legend,unemp_rate,delta_unemp
0,Johor,2017-01-01,5.978778,historical,4.400000,-0.650000
1,Johor,2018-01-01,5.988315,historical,3.300000,-1.100000
2,Johor,2019-01-01,3.589793,historical,3.550000,0.250000
3,Johor,2020-01-01,-3.751183,historical,4.500000,0.950000
4,Johor,2021-01-01,2.724879,historical,4.100000,-0.400000
...,...,...,...,...,...,...
219,W.P. Putrajaya,2026-01-01,-0.132383,forecast,1.233460,-0.012365
220,W.P. Putrajaya,2027-01-01,-0.132383,forecast,1.223568,-0.009892
221,W.P. Putrajaya,2028-01-01,-0.132383,forecast,1.215654,-0.007913
222,W.P. Putrajaya,2029-01-01,-0.132383,forecast,1.209324,-0.006331


In [4]:
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    state[["state", "state_id"]],
    on="state",
    how="left"
)


df_final = df.drop(columns=["date", "state"])
id_cols = ["date_id", "state_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [5]:
df_final["okun_id"] = ["OKUN" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["okun_id"] + [c for c in df_final.columns if c != "okun_id"]]
df_final

,okun_id,date_id,state_id,gdp_growth_yoy,legend,unemp_rate,delta_unemp
0,OKUN0001,DT005,ST001,5.978778,historical,4.400000,-0.650000
1,OKUN0002,DT009,ST001,5.988315,historical,3.300000,-1.100000
2,OKUN0003,DT013,ST001,3.589793,historical,3.550000,0.250000
3,OKUN0004,DT017,ST001,-3.751183,historical,4.500000,0.950000
4,OKUN0005,DT021,ST001,2.724879,historical,4.100000,-0.400000
...,...,...,...,...,...,...,...
219,OKUN0220,DT041,ST016,-0.132383,forecast,1.233460,-0.012365
220,OKUN0221,DT045,ST016,-0.132383,forecast,1.223568,-0.009892
221,OKUN0222,DT049,ST016,-0.132383,forecast,1.215654,-0.007913
222,OKUN0223,DT053,ST016,-0.132383,forecast,1.209324,-0.006331


In [6]:
write_table(df_final, "sc_gold", "fact_okun")

Table sc_gold.fact_okun written successfully.
